# Bias generator v3: new templates × retrieved examples (Kaggle)
Settings: **Accelerator → GPU T4 x2**, **Internet → On**, then **Save Version → Save & Run All (Commit)**.

A 2×2 comparison: model **v2** (already trained, included in the dataset) vs **v3** (retrained with new task templates), each with **0** or **2 retrieved examples** in the prompt. Download **results.zip** from the Output tab when it finishes.

In [ ]:
import glob, os, shutil
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
# pick the v3 dataset even if an older xhail-bias dataset is also attached
src = next(os.path.dirname(p) for p in glob.glob('/kaggle/input/**/pyproject.toml', recursive=True)
           if os.path.exists(os.path.join(os.path.dirname(p), 'experiments/bias_llm/data/sft_train_v3.jsonl')))
shutil.copytree(src, '/kaggle/working/xhail', dirs_exist_ok=True)
%cd /kaggle/working/xhail
!pip install -q unsloth
!pip install -q -e .
%cd experiments/bias_llm
!wc -l data/sft_train_v3.jsonl data/sft_val_v3.jsonl data/test.jsonl data/test_new.jsonl
!ls checkpoints/qwen-sft

## 1. Train v3 (≈80–90 min)

In [ ]:
if not os.path.exists('checkpoints/qwen-sft-v3/adapter_config.json'):
    !python train_unsloth.py --sft --sft-tag _v3 --model unsloth/Qwen2.5-1.5B-Instruct --epochs 2 --out checkpoints/qwen-sft-v3

## 2. Generate: 2 models × {0, 2} retrieved examples, up to 4 attempts (attempt 1 = single)
Tiers: **test** (original 160 synthetic), **new** (80 held-out tasks from the new templates), **gold** (16). 3 seeds each.

In [ ]:
tiers = {'test': 'data/test.jsonl', 'new': 'data/test_new.jsonl', 'gold': 'gold'}
runs = [('v2', 'checkpoints/qwen-sft', 0, ['new']),
        ('v2', 'checkpoints/qwen-sft', 2, ['test', 'new', 'gold']),
        ('v3', 'checkpoints/qwen-sft-v3', 0, ['test', 'new', 'gold']),
        ('v3', 'checkpoints/qwen-sft-v3', 2, ['test', 'new', 'gold'])]
for name, model, shots, use in runs:
    for tier in use:
        for seed in [0, 1, 2]:
            out = f'preds/v3/{name}_shots{shots}_{tier}_resample_s{seed}.jsonl'
            if not os.path.exists(out):
                !python colab_eval.py --model {model} --tasks {tiers[tier]} --strategy resample --shots {shots} --shot-pool data/train_v3.jsonl --seed {seed} --out {out}
# sanity floor for v3 on programs it was trained on (80 old + 80 new families)
for tier, tasks in [('seen_old', 'data/train.jsonl --limit 80'), ('seen_new', 'data/train_new.jsonl --limit 80')]:
    out = f'preds/v3/v3_shots0_{tier}_resample_s0.jsonl'
    if not os.path.exists(out):
        !python colab_eval.py --model checkpoints/qwen-sft-v3 --tasks {tasks} --strategy resample --seed 0 --out {out}

## 3. Repair with v3 (gold and new tiers)

In [ ]:
for tier, path in [('gold', 'data/repair_gold.jsonl'), ('new', 'data/repair_test_new.jsonl')]:
    out = f'preds/v3/v3_repair_{tier}.jsonl'
    if not os.path.exists(out):
        !python colab_eval.py --model checkpoints/qwen-sft-v3 --repair {path} --out {out}

## 4. Bundle results for download

In [ ]:
!cd /kaggle/working/xhail/experiments/bias_llm && zip -qr /kaggle/working/results_v3.zip preds/v3 checkpoints/qwen-sft-v3
!ls -la /kaggle/working/results_v3.zip